# Example: Prepare and run a single MESA model

In this example, we will see how to use the `morpy` package to prepare and run a single `MESA` model with `Python`.

We will look at two way to compute a `MESA` model: running and launching it. In the former case, we run the model in blocking mode (the `Python` function call finishes when the `MESA` model finishes running) and without status tracking. This is the simplest use case of `mesa-runner`'s `run` module. When we launch a model, the `Python` call immediately returns (non-blocking mode) and we can use `mesa-runner`'s status tracking capabilities to keep and eye on the model until it finishes.


## 1. Imports

In [ ]:
from pathlib import Path # We use Path objects instead of strings for cleaner-looking paths to fiels and directories 
import morpy as morpy # The mesa-runner package. We import the whole package for convenience, but separate impprts for the modules is also possible.
import pandas as pd # Even though pandas is imported automatically when installed and required by the package, we import it here to tweak the pandas optiosn for better-looking tables.

# Pandas tweaks for better looking tables
pd.set_option("display.max_rows", None) # Change this to None if you want to see complete tables. The maximum of 50 is used to keep the notebook output manageable. 
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)


## 2. General settings

The first few lines of the code block below are used to specify the `MESA` installation that you would like to use. These are similar to the environment variables you set in your `.*rc` files during the `MESA` and `MESASDK` installation (as specified here: https://docs.mesastar.org/en/26.4.1/installation.html) or in your module files. Even though the code has the capabilities to parse your `MESA` version from the `MESA` installation directory (here `MESA_DIR`), it does not hurt to specify it explicitly. Just make sure that the specified `MESA_VERSION` matches that of the `MESA` version installed at `MESA_DIR` to avoid conflicts.


The variable `PROJECT_DIR` is where your `MESA` project will be created. A `MESA` project consists of all the necessary files to compile the `Fortran` executables (`clean`, `mk`, `make/`, and `src/`), as well as the files to specify the hostory and profile output. Each `MESA` model gets its own directory within the project direcotory, where its inlists are stored and its output is written.


In [2]:
# Adjust these paths to match your local desired MESA installation.
MESA_DIR = Path('/Users/janhenneco/software/mesa/mesa-r24.08.1')
MESASDK_ROOT = Path('/Applications/mesasdk-25.12.1/')
OMP_THREADS  = 6 # Number of threads used per MESA model. Can be specified on a model-per-model basis when running/launching the models.
MESA_VERSION = 'r24.08.1'

# Path to the project directory. Adjust this to the desired location for storing your MESA project.
PROJECT_DIR = Path('/Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/prepare_and_run_single_model/project_dir')


## 3. Set up `MESA` environment and parse default option files

In this step, which you only have to do once per project, we do two things. 

First, we set up the `MESA` environment, just as you would do in your `.*rc` or module file (e.g. `export MESA_DIR=...`), and create a `MesaConfig` object (called `mconf` in the code below). The latter holds information about the `MESA` installation, such as the `MESA` version in a normalised numeric format allowing the version-aware parts of the code to work properly. The `setup_mesa_env` function is a wrapper function calling two functions that can alsoe be used separately (e.g. when you only want to create `MESA` inlists and do no need to compile it and run models, you only need to `MesaConfig` object): MesaConfig.setup() for the MesaConfig object and MesaConfig.set_mesa_env() for a dictionary containing the runtime environment for `MESA`.

Second, we parse the `MESA` default options files. These files usually live at `MESA_DIR/<star|binary|kap|eos>/defaults`. The resulting `DefaultsOptions` file (called `def_opts` here) contains `DefaultsFile` objects for all the different available default files in the `MESA` installation directory. These `DefaultsFile` objects are can be explored, which comes in handy for creating inlists (see later).

Although many of the methods and functions used in the `mesa-runner` package can generate the `MesaConfig` and `DefaultsOptions` objects on the fly, it is recommended to generate them once and pass them as an argument when a function or method requires it to avoid to much overhead.

In [ ]:
mconf = morpy.setup_mesa_env(mesa_dir=MESA_DIR, mesasdk_root=MESASDK_ROOT, omp_num_threads=OMP_THREADS, version_name=MESA_VERSION)
def_opts = morpy.build_defaults_options_from_config(mconf)
print('MESA version:', mconf.mesa_version.version_name)
print('     - year:', mconf.mesa_version.year)
print('     - month:', mconf.mesa_version.month)
print('     - patch:', mconf.mesa_version.patch)
print('     - scheme:', mconf.mesa_version.scheme)
print('     - revision:', mconf.mesa_version.revision)

MESA version: r24.08.1
     - year: 24
     - month: 8
     - patch: 1
     - scheme: release
     - revision: None


## 4. Exploring the `MESA` default options

In this step, we see how we can display and search the `DefaultsFile` objects contained in the `DefaultsOptions` object generated above.

Running the cell below will print out the attributes of the `DefaultsFile` called `controls`, which is itself an attribute of the `DefaultsOptions` object. As the same suggests, this `def_opts.controls` holds the default options found in the `controls.defaults` file found at `MESA_DIR/star/defaults`. Each of the available `.defaults` files in `MESA` has been added as a `DefaultsFile` object to the `DefaultsOptions` object by the `build_defaults_options_from_config` function above.

In [ ]:
vars(def_opts.controls) # .star_job, .binary_controls, .binary_job, .pgstar, .kap ...

To read and consult the options (which are all `MesaOption` objects) part of the controls namelist, we can use the `preview` function as shown below. Note that the best way to use this function is by using the `pandas` package, which is listed as an optional package for the `mesa-runner` package. If `pandas` is not available, the `preview` function will display the options as plain text.

In [ ]:
morpy.preview(def_opts.star_job.options, max_desc=int(1e6), prefer_pandas=True) # `max_desc` is the maximum numbers of characters shown for the option descriptions.

A few things to note from the `preview` output:

1) The column with the current values ("current") is empty. This is where the values specified in the inlist will be listed (see later).
2) The numeric values in the table are in `Fortran` format (using `MESA`'s 'd' notation for double-point precision). The `MesaOption` object has a method to convert its `default_option` and `current_value` attributes to Python datatypes, as demostrated below: 

In [ ]:
initial_z_mesa = def_opts.controls.options['initial_z']
print('initial_z in MESA/Fortran format: ', initial_z_mesa.default_option, '   ---   type = ', type(initial_z_mesa.default_option))
initial_z = def_opts.controls.options['initial_z'].to_python_value(use_current=False)
print('initial_z in Python format: ', initial_z, '   ---   type = ', type(initial_z), '\n')

bool_mesa = def_opts.controls.options['do_conv_premix']
print('boolean in MESA/Fortran format: ', bool_mesa.default_option, '   ---   type = ', type(bool_mesa.default_option))
bool_val = def_opts.controls.options['do_conv_premix'].to_python_value(use_current=False)
print('boolean in Python format: ', bool_val, '   ---   type = ', type(bool_val), '\n')

str_mesa = def_opts.controls.options['thermohaline_option']
print('string in MESA/Fortran format: ', str_mesa.default_option, '   ---   type = ', type(str_mesa.default_option))
str_val = def_opts.controls.options['thermohaline_option'].to_python_value(use_current=False)
print('string in Python format: ', str_val, '   ---   type = ', type(str_val))


Let's say we want to learn about the options for wind mass loss in `MESA`, for example because we want to include that in our models by adding it to our inlist (see later). For that we can use the `DefaultsFile.search` function, as shown below. This function will look for the specified string in the option names and descriptions. 

In [ ]:
search_results = def_opts.controls.search('wind')
morpy.preview(search_results, max_desc=int(1e6), prefer_pandas=True)

If for some reason `pandas` is not available, we can also render this table as plain text, as mentioned before.

In [ ]:
search_results = def_opts.controls.search('wind')
table = morpy.preview(search_results, max_desc=int(1e6), prefer_pandas=False)
print(table)

If we just want to list all the option in `controls` that have a certain pattern in their name, we can use the `DefaultsFile.option_names` function, as shown below:

In [ ]:
search_results = def_opts.controls.option_names('wind')
for name in search_results:
    print(name)

To know more about one of these options, for example "hot_wind_scheme", we can use the `DefaultsFile.describe` function:

In [ ]:
description = def_opts.controls.describe('hot_wind_scheme')
print(description)

## 5. Creating, importing, and changing inlists

Now, we can start setting up our `MESA` model. Before we set up the project and in- and output directories, we need to specify what we want our model to do in the inlists.

`mesa-runner` gives you the option to construct your inlists entirely from within `Python`. Alternatively, you can also import an existing inlist file and make changes to the inlist from within Python.

> **DISCLAIMER**: 
> `mesa-runner` is not and will never be designed to replace learning how to construct working and sensible `MESA` models (i.e. inlists). This is and will remain the responsibility of the user. What `mesa-runner` does try to do, is make it easier to set up `MESA` models without having to edit multiple files and consult many files in many different locations (e.g. the default options files).

We will show two ways to get started: creating inlists from scratch and importing and editing an existing inlist file.

### 5.1. Creating inlists from scratch

The code below shows how we can initialise an `Inlist` object and populate this object with `MesaOptions` (= adding options to the inlist).

In [ ]:
inlist_model = morpy.Inlist.empty(name='inlist_model') # Initiate the Inlist object

# Set a single option in the inlist
# Please not the following:
# - the Python floating-point value passed to 'value' will be converted to Fortran syntax
# - although it is optional to pass the previously-generated DefaultsOptions object (def_opts), 
#   it is recommended to avoid parsin the defaults files every time an option is added to the inlist
# - the code automatically recognises the namelist a certain option belongs to, but fails to add the
#   option if it is part of multiple namelists. Therefore, it is safer to pass the namelist explicitly. 
inlist_model.set_option(name='initial_z', value = 1.4e-2, defaults=def_opts, namelist='controls')
inlist_model.set_option(name='create_pre_main_sequence_model', value = True, defaults=def_opts, namelist='star_job')

to_print = morpy.preview(inlist_model.options, max_desc=int(1e6), prefer_pandas=False)
print(to_print)


We can keep going and populate our inlist like this if we want to, but it can be more efficient to set options in bulk per namelist. For that we can use the `Inlist.set_options` function.

In [ ]:
controls_options = {
   'num_trace_history_values': 2,
   'trace_history_value_name(1)': 'rel_E_err',
   'trace_history_value_name(2)': 'log_rel_run_E_err',
   'use_gold2_tolerances': True,
   'initial_mass': 20,
   'initial_z': '1d-5',
   'mixing_length_alpha': 2,
   'conv_core_gap_dq_limit': 1e-1,
   'xa_central_lower_limit_species(1)': 'h1',
   'xa_central_lower_limit(1) ': '0.01d0',
   'overshoot_scheme(1)': 'exponential',
   'overshoot_zone_type(1)': 'nonburn',
   'overshoot_zone_loc(1)': 'shell',
   'overshoot_bdy_loc(1)': 'bottom',
   'overshoot_f(1)': '0.128d0',
   'overshoot_f0(1)': '0.1d0',
   'overshoot_scheme(2)': 'exponential',
   'overshoot_zone_type(2)': 'any',
   'overshoot_zone_loc(2)': 'any',
   'overshoot_bdy_loc(2)': 'any',
   'overshoot_f(2) ': '0.014d0',
   'overshoot_f0(2)': '0.004d0',
   'photo_interval': 50,
   'profile_interval ':100,
   'history_interval': 10,
   'terminal_interval': 10,
   'write_header_frequency ': 100
   }
# Note that we can choose here: either use Fortran syntax and pass the values as strings, or Python syntax. 
# The latter will be converted into Fortran syntax automatically.

star_job_options = {
   'show_log_description_at_start': False,
   'load_saved_model': '.false.',
   'save_model_when_terminate': '.true.',
   'save_model_filename': 'final.mod',
   'required_termination_code_string': 'xa_central_lower_limit',
   'set_initial_cumulative_energy_error': True,
   'new_cumulative_energy_error': 0.0,
   'pgstar_flag': True,
   'save_pgstar_files_when_terminate': False,
   'initial_model_number': 0}

inlist_model.set_options(defaults=def_opts, controls= controls_options)
inlist_model.set_options(defaults=def_opts, star_job=star_job_options)

to_print = morpy.preview(inlist_model.options, max_desc=int(1e6), prefer_pandas=False)
print(to_print)

To get a working `MESA` model inlist setup, we still need a "header" inlist, which is typically called 'inlist'.

In [ ]:
inlist = morpy.Inlist.empty('inlist')

inlist.set_options(defaults=def_opts, 
                   star_job={'read_extra_star_job_inlist(1)': '.true.', 'extra_star_job_inlist_name(1)': 'inlist_model'},
                   controls={'read_extra_controls_inlist(1)': '.true.', 'extra_controls_inlist_name(1)': 'inlist_model'},
                   eos={'read_extra_eos_inlist(1)': '.true.', 'extra_eos_inlist_name(1)': 'inlist_model'},
                   kap={'read_extra_kap_inlist(1)': '.true.', 'extra_kap_inlist_name(1)': 'inlist_model'},
                   pgstar={'read_extra_pgstar_inlist(1)': '.true.', 'extra_pgstar_inlist_name(1)': 'inlist_model'})

to_print = morpy.preview(inlist.options, max_desc=int(1e6), prefer_pandas=False)
print(to_print)

### 5.2. Importing inlists from file

We also have the option to import inlist files to generate `Inlist` objects automatically, which is especially conventient when we want to start from a template inlist or vary on or more inlist options to for grid computations (see other examples).

In this example, we will import the inlist from the `MESA` test suite for high-mass stars, and make some changes to them.

In [ ]:
inlist_model = morpy.parse_inlist(MESA_DIR / "star" / "test_suite" /"high_mass" / "inlist_high_mass", 
                                 config=mconf, 
                                 default_options=def_opts, 
                                 new_name='inlist_model') # by default, the inlist filename (inlist_high_mass) 
                                                          # is used for Inlist.name, but we can change this with this argument
inlist_model.set_option(name='create_pre_main_sequence_model', value = True, defaults=def_opts, namelist='star_job')
inlist_model.set_option(name='load_saved_model', value = False, defaults=def_opts, namelist='star_job')

inlist_header = morpy.parse_inlist(MESA_DIR / "star" / "test_suite" /"high_mass" / "inlist", 
                                 config=mconf, 
                                 default_options=def_opts, 
                                 new_name='inlist')

to_print = morpy.preview(inlist_header.options, prefer_pandas=False)
print('Original inlist_header:')
print(to_print,'\n')

# Now that we have called out inlist_model something else, we need to make some quick changes to the header inlist
inlist_header.remove_option('read_extra_star_job_inlist(2)')
inlist_header.remove_option('extra_star_job_inlist_name(2)')
inlist_header.set_options(defaults=def_opts, 
                   star_job={'read_extra_star_job_inlist(1)': '.true.', 'extra_star_job_inlist_name(1)': 'inlist_model'},
                   controls={'read_extra_controls_inlist(1)': '.true.', 'extra_controls_inlist_name(1)': 'inlist_model'},
                   eos={'read_extra_eos_inlist(1)': '.true.', 'extra_eos_inlist_name(1)': 'inlist_model'},
                   kap={'read_extra_kap_inlist(1)': '.true.', 'extra_kap_inlist_name(1)': 'inlist_model'},
                   pgstar={'read_extra_pgstar_inlist(1)': '.true.', 'extra_pgstar_inlist_name(1)': 'inlist_model'})

to_print = morpy.preview(inlist_header.options, prefer_pandas=False)
print('Adjusted inlist_header:')
print(to_print)


Original inlist_header:
namelist | name                          | default     | current            | description                                                 
---------+-------------------------------+-------------+--------------------+-------------------------------------------------------------
star_job | read_extra_star_job_inlist(2) | .false.     | .true.             | You can split your ``star_job`` inlist into pieces using the following controls. BTW: it works re...
star_job | extra_star_job_inlist_name(2) | 'undefined' | 'inlist_high_mass' |                                                             
eos      | read_extra_eos_inlist(1)      | .false.     | .true.             | One can split an eos inlist into pieces using the following parameters. It works recursively, so ...
eos      | extra_eos_inlist_name(1)      | 'undefined' | 'inlist_high_mass' |                                                             
kap      | read_extra_kap_inlist(1)      | .false.     | .true

In [ ]:
to_print = morpy.preview(inlist_model.options, prefer_pandas=False)
print(to_print)

namelist | name                                | default     | current                  | description                                                 
---------+-------------------------------------+-------------+--------------------------+-------------------------------------------------------------
star_job | show_log_description_at_start       | .true.      | .false.                  | set this false if you want to skip the initial terminal output
star_job | load_saved_model                    | .false.     | .false.                  | By default at the start of a run a zams starting model is loaded, and then the ``initial_mass``, ...
star_job | load_model_filename                 | 'undefined' | 'zams.mod'               |                                                             
star_job | save_model_when_terminate           | .false.     | .true.                   | save final model when a run terminates only happens if satisfy required termination code if strin...
star_job | s

## 6. Adding inlists to a model and creating model and project directories

Now that we have two inlists ready to go, we need to add them to a Model object.

In [ ]:
hm_model = morpy.Model.empty('high_mass_model')

hm_model.add_inlist(inlist_header)
hm_model.add_inlist(inlist_model)

vars(hm_model)

>IMPORTANT INFORMATION ABOUT COPYING: Both Inlist and Model objects can be copied, which is especially conventient for setting up grid computations. Since both are instances, simply doing, for example

```
new_model = hm_model
hm_model.inlists['inlist_model'].set_option(name='initial_mass', value = 20, defaults=def_opts, namelist='controls')
```

> will change the inlist option for initial_mass in both models. To avoid this undesirable behaviour, use the deepcopy instance method Model.copy() or Inlist.copy():

```
new_model = hm_model.copy(model_copy_name = 'inlist_less_massive')
```

Now that we have a Model object, we are almost ready to run it. First, we need to write the contents of the Model object to the storage, i.e. create a model directory (as you would do for any `MESA` model) with the inlists and execution scripts (`rn` and `re`), and where the output will be written (at least by default if no custom output path are specified in the inlists).

Before we can do that, we need a project directory. This is the directory where the `MESA` compilation scripts and source files (i.e. the `src/` directory) live, and to which the `MESA` executables (`star` or `binary`) will be written. Although in the most basic `MESA` model setup one would have all of these files and directories in one big work directory, we chose to separate then into project and model directories. This design choice is evident for grid computations, which are essentially extensions of single-model computations. If we later want to create one work directory with all the necessary files and directories to, for example, share a workdir with somneone so they can reproduce your model, we can use the `work_setup.export_modeldir()` function (see later).

The create_projectdir will copy all the necessary files and directories from the `MESA` installation directory for a given mode ('star' or 'binary'). If you have a custom `run_star_extras.f` and/or `run_binary_extras.f` setup contained in an `src` directory somewhere, yiou can specify its path in `src_path` and this `src` directory will replace the standard one.

Since we already specified a `PROJECT_DIR` path above, we can go ahead and create a project directory in our system's storage:

In [ ]:
project_dir = morpy.create_projectdir(projectdir_path=PROJECT_DIR, mode='star') # mode = star because we want to create a single-star model
print(project_dir)

hm_model_dir = morpy.create_modeldir(model=hm_model, projectdir_path=project_dir, config=mconf, overwrite=True) # If you made a mistake and want
                                                                                               # to overwrite you model_dir, 
                                                                                               # use overwrite = True
print(hm_model_dir)

# Use this if you want to create single workdir for sharing and standard MESA runs:
exported_path = morpy.export_modeldir(model_dir=hm_model_dir, 
                     export_dir="/Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/prepare_and_run_single_model/exported_workdir",
                     projectdir_path=project_dir, 
                     overwrite=True)
print(exported_path)


/Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/prepare_and_run_single_model/project_dir
/Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/prepare_and_run_single_model/project_dir/high_mass_model
/Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/prepare_and_run_single_model/exported_workdir


## 7. Compiling the `MESA` executable

Now the the project and model directory are set up, we can go ahead and compile the `MESA` executable, in this case `star`.

We can again do this without leaving our current `Python` project by using the `compile_project()` function.
This function returns a CompileResult object, which contains all the information about the success, duration, and terminal output of the compile command, which we can inspect nicely using the `CompileResult.pretty()` function, as demonstrated below.

In [ ]:
compile_out = morpy.compile_project(projectdir_path=project_dir, mode="star")
print(compile_out.pretty(2000))

success: True
return_code: 0
duration_seconds: 2.2365472500096075
mode: star
executable_path: /Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/prepare_and_run_single_model/project_dir/star
model_scripts_updated: ['/Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/prepare_and_run_single_model/project_dir/high_mass_model/rn', '/Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/prepare_and_run_single_model/project_dir/high_mass_model/re']
clean_command_path: /Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/prepare_and_run_single_model/project_dir/clean
clean_return_code: 0
clean_duration_seconds: 0.20323912499588914
clean_stdout: 
clean_stderr: 
mk_command_path: /Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/prepare_and_run_single_model/project_dir/mk
mk_return_code: 0
mk_duration_seconds: 2.0329645839810837
mk_stdout: gfortran -Wno-uninitialized -fno-range-check -fmax-errors=7 -fdiagnostics-color  -fprote

## 8. Running or launching the model

In the first example, we'll run our `MESA` model from within `Python`, which means we have to wait until the model finishes before the `Python` call returns. In the second example, we'll launch the model and see how we can track its progress and outcome.

### 8.1. Running a model

To see if the model is actually running, you can use `top` or `htop` in a separate terminal session and look for a process calles `./star`. You can also locate the `log.out` file in the model directory, which shows the termianl output (this does not update line-per-line, unfortunately) or see live updates to the `log.out` file via `tail -f /path/to/model/log.out`. 

In [ ]:
result = morpy.run_model(hm_model, 
               default_options=def_opts, 
               config=mconf, 
               command_name='rn' # This option allows us to use a custom-named `rn` script or restart the model by passing `re`.
               )
vars(result)

ModelPathMissingError: Model 'loose_model' has no model_path. Create/export a model directory first.

The `result` variable holds a `RunResult` object, which, similarly to the `CompileResult` object holds information about the model run. This allows us, for example, to print out terminal output generated during the `MESA` run, as shown below:

In [ ]:
print(result.stdout)

### 8.2. Launching a model

The function to launch a model in a non-blocking way, `launch_model_process()` will spawn a process that runs the `MESA` model and immediately returns a tuple containing a `RunHandle` and process object. This object contains information about the launched process and allows us to track and "catch" it again. The function to "catch" the process again, is `wait_for_model_process()`. Once we run this function, we need to wait until the `MESA` model finishes again. This is essentially what happens in the `run_model()` function, which is a wrapper for `launch_model_process()` and `wait_for_model_process()`.

At first sight, it might seems useless to use `launch_model_process()` and `wait_for_model_process()` separately. Although partly true, these functions form the basis of the model grid running cababilities of `mesa-runner`. Another use case is that in which we want to launch a few models in parallel without using the full grid running capabilities.

In [ ]:
handle, process = morpy.launch_model(model=hm_model, default_options=def_opts, config=mconf, )

In [ ]:
result = morpy.wait_for_model(model=hm_model, handle=handle, process=process)
vars(result)